# EBM Quickstart: `EBM0D` and `EBM1DLat`

A bare-bones walkthrough of the two EBM variants in `paleobeasts.signal_models.ebm`.
The goal is to show how to instantiate and run each model out of the box, and to explain
the key parameter choices — especially `albedo`.
This is not a science notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import climatecritters as pb
from climatecritters.signal_models import ebm
from climatecritters.signal_models.ebm import EBM0D, EBM1DLat, albedo_func, OLR_func

## Class hierarchy

```
PBModel  (paleobeasts.core)
└── EBMBase
    ├── EBM0D      — 0D global-mean; step-by-step diagnostics
    └── EBM1DLat   — diffusive latitudinal grid; post-history diagnostics
```

`EBMBase` defines three shared helpers, each with signature `(self, T, t)`:

| Method | Default behaviour |
|--------|-------------------|
| `calc_OLR(T, t)` | Resolves `param_values['OLR']` — accepts a constant, callable, or `pb.Forcing`. |
| `calc_albedo(T, t)` | Resolves `param_values['albedo']`. |
| `calc_C(T, t)` | Resolves `param_values['C']`. |

**`EBM0D`** inherits all three defaults. Its `param_values` holds `'OLR'`, `'albedo'`, and `'C'`,
so any of them can be a float, a callable `(t, state)`, or a `pb.Forcing` at construction time.

**`EBM1DLat`** *overrides* `calc_OLR` (Budyko linear, array-valued) and `calc_albedo`
(array ice-line step function) because its physics differ from the scalar 0D formulation.
It inherits `calc_C` unchanged.

---

## `EBM0D` — zero-dimensional energy balance

$$C \frac{dT}{dt} = \frac{(1-\alpha)\,S_0}{4} - \mathrm{OLR}(T)$$

Temperature `T` is global-mean surface temperature in **Kelvin**.
`forcing` provides $S_0$ (W m⁻²) at every timestep.

| Parameter | Default | Notes |
|-----------|---------|-------|
| `forcing` | *required* | `pb.Forcing` returning $S_0$ (W m⁻²). |
| `OLR` | `OLR_func()` | Stefan-Boltzmann with dry-adiabat scaling (pRad=650, ps=1000). Adjust via `OLR_func(pRad=..., ps=...)`. |
| `C` | 4 | Heat capacity (W yr m⁻² K⁻¹). |
| `albedo` | 0.3 | Planetary albedo — accepts a `float`, callable, or `pb.Forcing`. See below. |
| `var_name` | `'temperature'` | Label stored on the output. |

In [ ]:
# Minimal instantiation: constant solar forcing, all defaults
forcing = pb.core.Forcing(lambda t: 1360.0)  # constant S0 (W/m^2)

model = EBM0D(forcing=forcing)               # albedo=0.3, OLR=OLR_func() by default
output = model.integrate(t_span=(0, 2000), y0=[288.0], method='RK45')

T_eq = output.state_variables['T'][-1]
print(f'Equilibrium T:         {T_eq:.2f} K')
print(f'Diagnostics available: {list(output.diagnostic_variables.keys())}')

### Specifying `albedo`

The `albedo` parameter accepts three forms:

**1. Constant (`float`)** — `albedo=0.3`

Fixed planetary reflectivity regardless of temperature. The default. Use this when
ice-albedo feedback is not the focus; the model has a single equilibrium.

---

**2. Module function** — `albedo=albedo_func`

Temperature-dependent ice-albedo feedback built into `paleobeasts`:

- below `T1 = 260 K`: ice-covered (α = 0.6)
- above `T2 = 290 K`: ice-free (α = 0.3)
- quadratic blend in between

Pass it directly — the tuneable thresholds are keyword-only (`*`), so
`inspect` sees only `(t, state)` and the PBModel dispatcher accepts the bare function.
This gives the model **two stable equilibria**: a warm state and a snowball state
depending on initial conditions.

To customise the thresholds, wrap it in a lambda:
```python
albedo=lambda t, state: albedo_func(t, state, T1=255., T2=285.)
```

---

**3. Custom callable** — any `(t)`, `(t, state)`, or `(t, state, model)` function

Full flexibility. Useful for smooth analytical forms, orbital modulation,
or coupling to another variable. The first positional argument must be named `t` or `time`.

In [ ]:
# Constant albedo vs. albedo_func (two initial conditions to show bistability)
model_const = EBM0D(forcing=forcing, albedo=0.3)
model_cold  = EBM0D(forcing=forcing, albedo=albedo_func)  # albedo_func passed directly
model_warm  = EBM0D(forcing=forcing, albedo=albedo_func)

out_const = model_const.integrate(t_span=(0, 2000), y0=[288.0], method='RK45')
out_cold  = model_cold.integrate( t_span=(0, 2000), y0=[250.0], method='RK45')  # below T1=260 K
out_warm  = model_warm.integrate( t_span=(0, 2000), y0=[295.0], method='RK45')  # above T2=290 K

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(out_const.time, out_const.state_variables['T'],
        label='constant albedo (0.3)',             color='steelblue')
ax.plot(out_cold.time,  out_cold.state_variables['T'],
        label='albedo_func — cold start (250 K)',  color='royalblue', ls='--')
ax.plot(out_warm.time,  out_warm.state_variables['T'],
        label='albedo_func — warm start (295 K)',  color='firebrick', ls='--')
ax.axhline(260, color='gray', lw=0.8, ls=':', label='T1 / T2 thresholds')
ax.axhline(290, color='gray', lw=0.8, ls=':')
ax.set_xlabel('time')
ax.set_ylabel('T (K)')
ax.set_title('EBM0D: constant vs. temperature-dependent albedo')
ax.legend(fontsize=9)
plt.tight_layout()

In [ ]:
# Custom callable: smooth tanh transition centred on 275 K
def my_albedo(t, state):
    T = float(np.asarray(state).reshape(-1)[0])
    return 0.45 - 0.15 * np.tanh((T - 275.0) / 10.0)

model_custom = EBM0D(forcing=forcing, albedo=my_albedo)
out_custom = model_custom.integrate(t_span=(0, 2000), y0=[288.0], method='RK45')

T_eq_custom = out_custom.state_variables['T'][-1]
print(f'Equilibrium T (custom albedo): {T_eq_custom:.2f} K')

---
## `EBM1DLat` — diffusive latitudinal EBM

$$C \frac{\partial T}{\partial t} = S(x)\,(1-\alpha(T)) - \mathrm{OLR}(T) + D\,\nabla^2 T$$

where $x = \sin\varphi$. Temperature is in **°C** (initial conditions, Budyko OLR
coefficients, and albedo thresholds are all calibrated in Celsius).

### Key differences from `EBM0D`

| Feature | `EBM0D` | `EBM1DLat` |
|---------|---------|------------|
| Units | Kelvin | Celsius |
| OLR | Stefan-Boltzmann (from `param_values`) | Budyko linear: $(A - F_{\mathrm{CO_2}}) + B\,T$ (overrides `calc_OLR`) |
| Albedo | `param_values['albedo']` — swappable at construction | `calc_albedo` method — replace by subclassing |
| `forcing` | Required ($S_0$) | Optional; solar input is set by the `S0` parameter |
| Diagnostics | Accumulated in `dydt` | Derived from full trajectory after solve |

### Parameters

| Parameter | Default | Notes |
|-----------|---------|-------|
| `S0` | 1365.0 | Solar constant (W m⁻²). |
| `C` | 10.0 | Heat capacity. |
| `D` | 0.55 | Meridional diffusion coefficient. |
| `A` | 210.0 | Budyko OLR intercept (W m⁻²). |
| `B` | 2.0 | Budyko OLR slope (W m⁻² °C⁻¹). |
| `CO2_forcing` | 0.0 | Radiative forcing (W m⁻²) — subtracts from `A`, warming the climate. |
| `grid_n` | 50 | Number of latitude grid points (−90° to 90°). |

All parameters are registered in `param_values`, so any of them can be a callable
or `pb.Forcing`. For example, a CO₂ ramp:
```python
co2_ramp = pb.core.Forcing.from_sequence([...])
model = EBM1DLat(CO2_forcing=co2_ramp)
```

### Albedo in `EBM1DLat`

Unlike `EBM0D`, albedo is not in `param_values`. `EBM1DLat.calc_albedo(T, t)` computes
an **array** of albedo values from the local temperature at each grid point:

- T < −10 °C → α = 0.6
- T > 0 °C → α = 0.3
- linear blend in between

To replace the albedo scheme, subclass `EBM1DLat` and override `calc_albedo(self, T, t)`.
The module also provides `albedo_func1D(t, state, model, *, ...)` for a
P₂ Legendre-polynomial alternative — use it by subclassing.

In [ ]:
# Minimal instantiation: no forcing object needed, S0 is a parameter
model1d = EBM1DLat(S0=1365.0, grid_n=50)
out1d = model1d.integrate(t_span=(0, 200), y0=[15.0], method='RK45')  # y0 in °C

print(f'Grid points:           {len(out1d.state_variables.dtype.names)}')
print(f'Equilibrium Tglobal:   {out1d.diagnostic_variables["Tglobal"][-1]:.2f} °C')
print(f'Equilibrium ice line:  {out1d.diagnostic_variables["ice_line_lat"][-1]:.1f}°')

In [ ]:
phi = model1d.phi  # latitude grid in degrees
T_final = np.array([out1d.state_variables[f'T_{i}'][-1] for i in range(model1d.grid_n)])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Final zonal-mean temperature profile
axes[0].plot(phi, T_final, color='steelblue')
axes[0].axhline(-10, color='royalblue', ls='--', lw=0.8, label='ice-line threshold (−10 °C)')
axes[0].set_xlabel('latitude (°)')
axes[0].set_ylabel('T (°C)')
axes[0].set_title('Final temperature profile')
axes[0].legend(fontsize=8)

# Global mean temperature vs. time
axes[1].plot(out1d.time, out1d.diagnostic_variables['Tglobal'], color='firebrick')
axes[1].set_xlabel('time')
axes[1].set_ylabel('Tglobal (°C)')
axes[1].set_title('Global mean temperature')

# Ice-line latitude vs. time
axes[2].plot(out1d.time, out1d.diagnostic_variables['ice_line_lat'], color='royalblue')
axes[2].set_xlabel('time')
axes[2].set_ylabel('latitude (°)')
axes[2].set_title('Ice-line latitude')

plt.suptitle('EBM1DLat — equilibration from uniform 15 °C', y=1.02)
plt.tight_layout()